# 04 — Phase 1: Rank Sweep  ⏱️ OFFLINE — pre-run before the demo

This is the heavy, pre-computed stage. For every compressible 3×3 conv it
sweeps a 2D grid of `(r_in, r_out)` ranks, and for each combination measures:

- **noise %** — the std-based activation-error metric (identical to the
  CNN/ViT pipeline), comparing the original conv's output to the Tucker
  block's output on a stratified sample.
- **mAP@0.5** — detection quality on a subsampled validation set (the
  object-detection analog of the CNN pipeline's top-1 accuracy).

It then fits a symmetrized degree-4 (biquadratic) polynomial of **mAP vs
noise** per layer. Everything is checkpointed per layer to JSON so this can
run offline on the DGX and the results carried to the demo machine — the
live demo only runs Phase 2 on these stored results.

**Methodology note:** this faithfully mirrors the validated CNN pipeline
(full deep-copy per combo, full Tucker + spatial folding, std-based noise,
symmetrized even-polynomial fit). The one task substitution is mAP@0.5 in
place of top-1 accuracy on the polynomial's y-axis.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath("../src"))
import torch
from torch.utils.data import DataLoader

from model import YOLOv3
from dataset import CocoSubsetDataset, yolo_collate_fn
from tucker_pipeline import get_compressible_layers, fit_biquadratic_polynomial
from tucker_phase1 import evaluate_map, run_phase1

with open("../checkpoints/run_config.json") as f:
    cfg = json.load(f)
CLASS_NAMES, IMG_SIZE, DATA_ROOT, BATCH_SIZE = cfg["class_names"], cfg["img_size"], cfg["data_root"], cfg["batch_size"]
NUM_CLASSES = len(CLASS_NAMES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## Load the trained baseline model

In [ ]:
model = YOLOv3(num_classes=NUM_CLASSES)
state = torch.load("../checkpoints/yolov3_best.pt", map_location=DEVICE)
model.load_state_dict(state["model_state"])
model.to(DEVICE).eval()
print("loaded trained baseline")

## Build the validation loader + stratified sample

The **val loader** feeds the (subsampled) mAP evaluation. The **stratified
sample** feeds the noise metric — a handful of images per class is plenty,
since each image contributes hundreds of thousands of activation values to
the std-based noise estimate (same rationale as the CNN pipeline's 1-per-class).

In [ ]:
val_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                            images_per_class=30, augment=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=yolo_collate_fn, num_workers=0)

# stratified sample for noise: reuse a few val images per class
strat_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                              images_per_class=2, augment=False)
X_sample = torch.stack([strat_ds[i][0] for i in range(len(strat_ds))], dim=0)
print("stratified sample:", tuple(X_sample.shape))

## Baseline mAP (uncompressed reference)

Measured once up front — Phase 2 uses this as the reference the accuracy budget is subtracted from.

In [ ]:
baseline_map = evaluate_map(model, val_loader, NUM_CLASSES, DEVICE, IMG_SIZE, max_batches=None)
print(f"baseline mAP@0.5: {baseline_map:.4f}")

os.makedirs("../checkpoints/phase1", exist_ok=True)
with open("../checkpoints/phase1/baseline.json", "w") as f:
    json.dump({"baseline_map": baseline_map}, f)

## Enumerate compressible layers (37 for YOLOv3: 3×3 convs, skip stem + prediction heads)

In [ ]:
compressible = get_compressible_layers(model, skip_first=True)
print(f"{len(compressible)} compressible layers")
total = 0
for l in compressible:
    step = max(1, int(round(0.20 * min(l["C_in"], l["C_out"]))))
    import math
    n_in = len(list(range(step, l["C_in"]+1, step))) + (1 if l["C_in"] % step else 0)
    n_out = len(list(range(step, l["C_out"]+1, step))) + (1 if l["C_out"] % step else 0)
    total += n_in * n_out
print(f"~{total} total (r_in, r_out) combinations across all layers")

## Run Phase 1

⚠️ **This is the long-running cell.** Faithful to the CNN pipeline, it
deep-copies the full model per combination and evaluates mAP + noise. On the
DGX GPU this is feasible offline but not fast — leave it running. It
checkpoints after every layer to `../checkpoints/phase1/phase1_layerNNN.json`
and **resumes automatically** if interrupted (skips layers already done).

Tune `MAP_MAX_BATCHES` down to speed up the sweep (fewer val batches per mAP
estimate) or up for more accurate curves. Since Phase 1 is pre-run offline,
you can afford a reasonably large value here for trustworthy polynomials.

In [ ]:
MAP_MAX_BATCHES = 20   # val batches per mAP estimate (subsampling for tractability)
NOISE_BATCH_SIZE = 8

layer_results = run_phase1(
    model, compressible, X_sample, val_loader, NUM_CLASSES, DEVICE, IMG_SIZE,
    ckpt_dir="../checkpoints/phase1",
    map_max_batches=MAP_MAX_BATCHES, noise_batch_size=NOISE_BATCH_SIZE)
print(f"\nPhase 1 done: {len(layer_results)} layers")

## Quick look: noise-vs-mAP curve for one layer

Sanity-check that the fitted biquadratic tracks the swept points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

li = next(l for l in layer_results if l["poly_coeffs"] is not None)
sweep = [r for r in li["sweep_results"] if not np.isnan(r["noise_percent"])]
noise = [r["noise_percent"] for r in sweep]
mAP = [r["mAP"] for r in sweep]

xs = np.linspace(0, max(noise), 200)
poly = np.poly1d(li["poly_coeffs"])
plt.figure(figsize=(8, 5))
plt.scatter(noise, mAP, alpha=0.6, label="swept combos")
plt.plot(xs, poly(xs), "r-", label="biquadratic fit")
plt.xlabel("noise %"); plt.ylabel("mAP@0.5"); plt.title(f"Layer: {li['name']}")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Output

`../checkpoints/phase1/phase1_layerNNN.json` (one per layer) + `baseline.json`.
These are the only things Phase 2 needs — carry this folder to the demo machine.